In [11]:
!pip install pandas openpyxl
!pip install csvkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.9/95.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.6/114.6 kB 6.5 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
# 첫 번째 시트 변환
!in2csv "/content/drive/MyDrive/(데이터)(신장르)(대중음악포함)_19년 하반기~_23년 1분기 raw데이터_2022_01_01_10_676,444.xlsx" \
> "/content/drive/MyDrive/result/temp.csv"

# 특정 시트명이 있을 때 (예: '원본데이터')
# in2csv --sheet "원본데이터" "파일경로.xlsx" > "/content/drive/MyDrive/result/temp.csv"


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [4]:


# 1) 경로 설정
INPUT_XLSX = '/content/drive/MyDrive/(데이터)(신장르)(대중음악포함)_19년 하반기~_23년 1분기 raw데이터_2022_01_01_10_676,444.xlsx'  # 수정
OUTPUT_CSV  = '/content/drive/MyDrive/result/output_filtered.csv'  # 수정
OUTPUT_XLSX = '/content/drive/MyDrive/result/output_filtered.xlsx' # (선택)

# 2) 필요한 컬럼만 지정
cols = [
    '공연명','장르명','세부장르명','출연진내용','제작진내용','기획제작사명','원작자명','극작가명'
]

# 3) 엑셀 로드 (전부 문자열로 읽기 권장: 엑셀 자동형변환 방지)
df = pd.read_excel(INPUT_XLSX, dtype=str, engine='openpyxl')

# 4) 컬럼 정리: 앞뒤 공백 제거, 누락 컬럼은 만들어두기
df.columns = df.columns.str.strip()
for c in cols:
    if c not in df.columns:
        df[c] = ''  # 없는 컬럼은 빈 문자열 컬럼으로 채워둠

# 5) 필요한 컬럼만 선택
df = df[cols].copy()

# 6) 장르명이 '복합'인 행만 필터 (부분일치도 통과: '복합/…' 등)
df['장르명'] = df['장르명'].astype(str).str.strip()
df = df[df['장르명'].str.contains('복합', na=False)]

# 7) 공연명 중복 제거 (중복은 첫 번째만 남김)
df['공연명'] = df['공연명'].astype(str).str.strip()
df = df.drop_duplicates(subset=['공연명'], keep='first')

# (참고) 중복된 공연명을 '전부 제외'하려면 아래 두 줄로 교체
# dup_names = df['공연명'][df['공연명'].duplicated(keep=False)].unique()
# df = df[~df['공연명'].isin(dup_names)]

# 8) 저장 (엑셀/CSV)
df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')  # 엑셀에서 바로 열 거면 utf-8-sig 권장
with pd.ExcelWriter(OUTPUT_XLSX, engine='openpyxl') as w:
    df.to_excel(w, index=False)
print('done:', len(df))


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


KeyboardInterrupt: 